## Installation (à exécuter une seule fois)

In [1]:
# install.packages("reticulate")
# install.packages("data.table")
# install.packages("remotes")
# remotes::install_cran("iml")
# remotes::install_github("dandls/counterfactuals")

## Imports

In [2]:
library(reticulate)
library(iml)
library(counterfactuals)
library(data.table)

# IMPORTANT : pointer vers le python de TON environnement virtuel du projet
# (gradual-counterfactuals/venv), pas un python quelconque du systeme.
use_python("D:/M2/Soutenance/Experimentations/gradual-counterfactuals/venv/Scripts/python.exe", required = TRUE)
joblib <- import("joblib")

# On supprime le warning inoffensif "X does not have valid feature names"
# (le modele a ete entraine avec un DataFrame nomme, on lui passe ici une matrice)
py_run_string("import warnings; warnings.filterwarnings('ignore', message='.*valid feature names.*')")

cat("Bibliotheques importees.\n")

Bibliotheques importees.


## Paramètres

In [3]:
TARGET_COL     <- "price"

# IMPORTANT : ces deux valeurs doivent correspondre exactement a celles
# utilisees dans 3_Contrefactuels.ipynb (Cellule 6) du pipeline Python.
INDEX_INSTANCE <- 22L    # attention : indexation R en base 1 (X_test[1,] = X_test.iloc[0] en Python)
DELTA          <- 1485.77

SEUIL_VALIDITE   <- 0.01
MARGE_GENERATION <- 0.05
N_GENERATIONS    <- 15L

## Chargement des fichiers (issus des notebooks 1 et 2)

In [4]:
OUTPUT_DIR <- "../../outputs/automobile"

model_py <- joblib$load(file.path(OUTPUT_DIR, "svm_model.pkl"))

X_train <- fread(file.path(OUTPUT_DIR, "X_train.csv"), sep = ";")
y_train <- fread(file.path(OUTPUT_DIR, "y_train.csv"), sep = ";")[[TARGET_COL]]
X_test  <- fread(file.path(OUTPUT_DIR, "X_test.csv"), sep = ";")

# Instance a expliquer : ligne INDEX_INSTANCE du test set
exemple <- X_test[INDEX_INSTANCE, ]

cat(sprintf("[MOC] %d lignes de train chargees, instance %d du test set selectionnee.\n",
            nrow(X_train), INDEX_INSTANCE-1))

[MOC] 128 lignes de train chargees, instance 21 du test set selectionnee.


## Fonction de prédiction (pont vers le modèle scikit-learn)

In [5]:
predict_fun <- function(model, newdata) {
  X_np  <- r_to_py(as.matrix(newdata))
  preds <- model$predict(X_np)
  data.frame(price = as.numeric(py_to_r(preds)))
}

fx      <- predict_fun(model_py, exemple)$price[1]
Y_CIBLE <- fx + DELTA

cat(sprintf("[MOC] Prediction f(x) : %.2f\n", fx))
cat(sprintf("[MOC] Cible y'        : %.2f\n", Y_CIBLE))

[MOC] Prediction f(x) : 10514.23
[MOC] Cible y'        : 12000.00


## Construction du Predictor (iml) et lancement de MOC

In [6]:
# Detection du sens (meme logique que le pipeline Python)
if (Y_CIBLE > fx) {
  SENS <- "+"
} else if (Y_CIBLE < fx) {
  SENS <- "-"
} else {
  stop("Y_CIBLE est egal a f(x) : aucune modification n'est necessaire.")
}

marge         <- MARGE_GENERATION * abs(Y_CIBLE)
epsilon_marge <- 1e-2  # garde-fou pour ne jamais inclure fx dans l'intervalle

if (SENS == "+") {
  borne_inf <- max(Y_CIBLE - marge, fx + epsilon_marge)
  borne_sup <- Y_CIBLE + marge
} else {
  borne_inf <- Y_CIBLE - marge
  borne_sup <- min(Y_CIBLE + marge, fx - epsilon_marge)
}

desired_outcome <- c(borne_inf, borne_sup)
cat(sprintf("[MOC] f(x) = %.2f | Y_CIBLE = %.2f | desired_outcome = [%.2f, %.2f]\n",
            fx, Y_CIBLE, desired_outcome[1], desired_outcome[2]))

# Predictor construit sur les donnees d'ENTRAINEMENT uniquement
predictor <- Predictor$new(
  model             = model_py,
  data              = X_train,
  y                 = y_train,
  predict.function  = predict_fun
)

cat("[MOC] Lancement de l'algorithme genetique NSGA-II...\n")
moc_regr <- MOCRegr$new(predictor, n_generations = N_GENERATIONS, quiet = TRUE)

cfactuals <- moc_regr$find_counterfactuals(
  x_interest      = as.data.frame(exemple),
  desired_outcome = desired_outcome
)

cat(sprintf("[MOC] %d contrefactuel(s) trouve(s) sur le front de Pareto.\n", nrow(cfactuals$data)))
cfactuals$data

[MOC] f(x) = 10514.23 | Y_CIBLE = 12000.00 | desired_outcome = [11400.00, 12600.00]
[MOC] Lancement de l'algorithme genetique NSGA-II...


`x_interest` was removed from results.



[MOC] 17 contrefactuel(s) trouve(s) sur le front de Pareto.


normalized_losses,wheel_base,length,width,height,curb_weight,engine_size,bore,stroke,compression_ratio,horsepower,peak_rpm,city_mpg,highway_mpg
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>
125.78149,103.3000,174.6,64.60000,59.8,2535,122,3.34,3.460000,8.500000,88,5000,20,30
110.00000,103.3000,174.6,67.29873,59.8,2535,122,3.34,3.460000,8.500000,88,5000,24,30
110.00000,103.3000,174.6,64.60000,59.8,2535,162,3.34,3.460000,8.500000,88,5000,24,30
110.00000,103.3000,174.6,66.18600,59.8,2535,122,3.34,3.460000,8.500000,88,5000,24,30
108.32496,103.3000,174.6,66.20846,59.8,2535,122,3.34,3.460000,8.500000,88,5000,24,30
110.00000,103.3000,174.6,65.92986,59.8,2535,122,3.34,3.460062,8.500000,88,5000,24,30
110.00000,103.3000,174.6,64.60000,59.8,2758,122,3.34,3.460000,8.743593,88,5000,24,30
110.00000,103.3000,174.6,64.60484,59.8,2758,122,3.34,3.460000,8.500000,88,5000,24,30
110.00000,103.8066,174.6,64.87370,59.8,2535,122,3.34,3.460000,8.004447,88,5000,24,30


## Affichage au format comparable

In [7]:
cf_df   <- as.data.frame(cfactuals$data)
x_orig  <- as.data.frame(exemple)[1, ]
cols    <- names(exemple)

rows <- list()

ligne_orig <- c(motif = "-", nb_sauts = "-", support = "-", pred_cible = "-")
for (col in cols) ligne_orig[col] <- round(x_orig[[col]], 4)
ligne_orig["price"] <- round(fx, 2)
rows[[1]] <- ligne_orig

for (i in seq_len(nrow(cf_df))) {
  pred_cf <- predict_fun(model_py, cf_df[i, cols, drop = FALSE])$price[1]

  ligne <- c(motif = "MOC (NSGA-II)", nb_sauts = "-", support = "-",
             pred_cible = paste0("[", round(desired_outcome[1],2), ", ", round(desired_outcome[2],2), "]"))
  for (col in cols) {
    val_cf <- cf_df[i, col]
    if (abs(x_orig[[col]] - val_cf) > 1e-6) {
      fleche <- if (val_cf > x_orig[[col]]) "↑" else "↓"
      ligne[col] <- paste0(round(val_cf, 4), " (", fleche, ")")
    } else {
      ligne[col] <- "-"
    }
  }
  fleche_price <- if (pred_cf > fx) "↑" else "↓"
  ligne["price"] <- paste0(round(pred_cf, 2), " (", fleche_price, ")")
  rows[[length(rows) + 1]] <- ligne
}

df_cf_moc <- as.data.frame(do.call(rbind, rows), stringsAsFactors = FALSE)
fwrite(df_cf_moc, file.path(OUTPUT_DIR, "contrefactuels_moc.csv"), sep = ";")

cat("[MOC] Resultats sauvegardes dans contrefactuels_moc.csv\n")
print(df_cf_moc)

[MOC] Resultats sauvegardes dans contrefactuels_moc.csv
           motif nb_sauts support     pred_cible normalized_losses   wheel_base
1              -        -       -              -               110        103.3
2  MOC (NSGA-II)        -       - [11400, 12600]      125.7815 (↑)            -
3  MOC (NSGA-II)        -       - [11400, 12600]                 -            -
4  MOC (NSGA-II)        -       - [11400, 12600]                 -            -
5  MOC (NSGA-II)        -       - [11400, 12600]                 -            -
6  MOC (NSGA-II)        -       - [11400, 12600]       108.325 (↓)            -
7  MOC (NSGA-II)        -       - [11400, 12600]                 -            -
8  MOC (NSGA-II)        -       - [11400, 12600]                 -            -
9  MOC (NSGA-II)        -       - [11400, 12600]                 -            -
10 MOC (NSGA-II)        -       - [11400, 12600]                 - 103.8066 (↑)
11 MOC (NSGA-II)        -       - [11400, 12600]                

## Métriques

In [8]:
calculer_metriques <- function(x_orig, contrefactuels_list, fx_cible, seuil_validite = 0.01, mad) {
  n <- length(contrefactuels_list)
  if (n == 0) return(NULL)

  nb_valides <- sum(sapply(contrefactuels_list, function(cf) {
    abs(cf$pred_cf - fx_cible) < seuil_validite * abs(fx_cible)
  }))
  validite <- nb_valides / n

  proximites <- sapply(contrefactuels_list, function(cf) {
    mean(abs(unlist(x_orig) - unlist(cf$x_cf)) / mad)
  })
  proximite <- mean(proximites)

  nb_cols <- length(x_orig)
  sparsite_vals <- sapply(contrefactuels_list, function(cf) {
    nb_modifiees <- sum(abs(unlist(x_orig) - unlist(cf$x_cf)) > 1e-6)
    1 - nb_modifiees / nb_cols
  })
  sparsite <- mean(sparsite_vals)

  if (n < 2) {
    diversite <- 0
  } else {
    paires <- c()
    for (i in 1:(n - 1)) {
      for (j in (i + 1):n) {
        d <- mean(abs(unlist(contrefactuels_list[[i]]$x_cf) - unlist(contrefactuels_list[[j]]$x_cf)) / mad)
        paires <- c(paires, d)
      }
    }
    diversite <- mean(paires)
  }

  list(nb = n, validite = round(validite, 4), proximite = round(proximite, 4),
       sparsite = round(sparsite, 4), diversite = round(diversite, 4))
}

# MAD calcule sur x_orig (l'instance elle-meme), meme logique que la methode maison
mad <- sapply(x_orig[cols], function(v) {
  m <- median(abs(v - median(v)))
  if (m == 0) 1 else m
})

contrefactuels_list <- lapply(seq_len(nrow(cf_df)), function(i) {
  pred_cf <- predict_fun(model_py, cf_df[i, cols, drop = FALSE])$price[1]
  list(x_cf = cf_df[i, cols], pred_cf = pred_cf)
})

m <- calculer_metriques(x_orig[cols], contrefactuels_list, Y_CIBLE, SEUIL_VALIDITE, mad)

cat(strrep("=", 50), "\n")
cat("  METRIQUES DU CONTREFACTUEL (MOC)\n")
cat(strrep("-", 50), "\n")
for (k in names(m)) cat(sprintf("  %-12s: %s\n", k, m[[k]]))

  METRIQUES DU CONTREFACTUEL (MOC)
-------------------------------------------------- 
  nb          : 17
  validite    : 0
  proximite   : 2.2903
  sparsite    : 0.8782
  diversite   : 4.3039
